# Baseline Analysis

In [10]:
import polars as pl

from social_groups.reporting.analysis_columns import AnalysisColumn
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
)
from social_groups.reporting.parsing import (
    AnswerComparer,
    AnswerOptions,
    AnswerParser,
)


In [11]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling="wrong"
)

In [12]:
from social_groups.analysis.definitions import defs

baseline_frame = defs.load_fn().load_asset_value("baseline")

2026-02-17 19:40:31 +0800 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/baseline.parquet using PolarsParquetIOManager...


In [13]:
baseline_frame.head()

id,run_id,question_id,phoenix_span_url,run_identifier,final_answer,original_question_id,category,question,answer_string,model_name
i64,i64,i64,str,str,str,i64,str,str,str,str
0,0,0,"""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's tackle thi…",10556,"""computer science""","""Q: In building a linear regres…","""C""","""Qwen/Qwen3-4B"""
1,0,1,"""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's tackle thi…",5567,"""other""","""Q: An entity prepares its fina…","""C""","""Qwen/Qwen3-4B"""
2,0,2,"""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's try to fig…",1945,"""law""","""Q: A wealthy woman often wore …","""E""","""Qwen/Qwen3-4B"""
3,0,3,"""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's tackle thi…",2948,"""biology""","""Q: What is differential reprod…","""I""","""Qwen/Qwen3-4B"""
4,0,4,"""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, so the question …",6622,"""health""","""Q: What stable isotope is comm…","""J""","""Qwen/Qwen3-4B"""


### Number of unparsable answers

In [14]:
(baseline_frame.with_columns(
    parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
)
.group_by("model_name")
.agg(
    no_null=pl.col(AnalysisColumn.parsed_answer.value)
    .str.starts_with("___")
    .not_()
    .sum(),
    null_percentage=(
            pl.col(AnalysisColumn.parsed_answer.value).str.starts_with("___").mean()
            * 100
    ).round(2),
)
)


model_name,no_null,null_percentage
str,u32,f64
"""Qwen/Qwen3-14B""",93,7.0
"""Qwen/Qwen3-4B""",88,12.0
"""Qwen/Qwen3-0.6B""",90,10.0


### Accuracy per Model

In [15]:
(
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .with_columns(
        is_correct=comparer(
            pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
        )
    )
    .group_by("model_name")
    .agg(accuracy=pl.col("is_correct").mean())
    .sort(pl.col("model_name").str.extract(r"-(\d+\.?\d*)B", 1).cast(pl.Float64))
)

model_name,accuracy
str,f64
"""Qwen/Qwen3-0.6B""",0.35
"""Qwen/Qwen3-4B""",0.59
"""Qwen/Qwen3-14B""",0.64
